In [ ]:
# Instalar librerías necesarias
!pip install scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ Librerías listas")

In [ ]:
# Cargamos el dataset directamente desde URL (sin necesidad de Kaggle)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)
print("✅ Dataset cargado")
print(df.shape)
df.head()

In [ ]:
# Ver información general
print(df.info())
print("\n")
print(df['Churn'].value_counts())

# Visualizar distribución de abandono
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribución de Clientes: Churn vs No Churn')
plt.show()

In [ ]:
# Convertir TotalCharges a numérico
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Eliminar filas con valores nulos
df.dropna(inplace=True)

# Eliminar columna ID (no aporta al modelo)
df.drop('customerID', axis=1, inplace=True)

print("✅ Datos limpios")
print(df.shape)

In [ ]:
# Convertir columnas de texto a números
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

print("✅ Variables codificadas")
df.head()

In [ ]:
# Separar características (X) y variable objetivo (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Dividir en 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Entrenamiento: {X_train.shape[0]} filas")
print(f"✅ Prueba: {X_test.shape[0]} filas")

In [ ]:
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train, y_train)

pred_lr = modelo_lr.predict(X_test)

print("📊 REGRESIÓN LOGÍSTICA")
print(f"Accuracy: {accuracy_score(y_test, pred_lr):.2%}")
print(classification_report(y_test, pred_lr))

In [ ]:
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

pred_rf = modelo_rf.predict(X_test)

print("📊 RANDOM FOREST")
print(f"Accuracy: {accuracy_score(y_test, pred_rf):.2%}")
print(classification_report(y_test, pred_rf))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matriz Regresión Logística
sns.heatmap(confusion_matrix(y_test, pred_lr),
            annot=True, fmt='d', ax=axes[0],
            cmap='Blues')
axes[0].set_title('Regresión Logística')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

# Matriz Random Forest
sns.heatmap(confusion_matrix(y_test, pred_rf),
            annot=True, fmt='d', ax=axes[1],
            cmap='Greens')
axes[1].set_title('Random Forest')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

In [ ]:
importancias = pd.Series(
    modelo_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=importancias.values, y=importancias.index, palette='viridis')
plt.title('Top 10 Variables que Predicen el Abandono')
plt.xlabel('Importancia')
plt.show()

In [ ]:
print("=" * 50)
print("RESUMEN DEL PROYECTO")
print("=" * 50)
print(f"Regresión Logística - Accuracy: {accuracy_score(y_test, pred_lr):.2%}")
print(f"Random Forest       - Accuracy: {accuracy_score(y_test, pred_rf):.2%}")
print()
print("Variable más importante para predecir abandono:")
print(f"👉 {importancias.index[0]}")
print()
print("Conclusión: El modelo Random Forest es más preciso")
print("para identificar clientes en riesgo de abandonar.")